In [1]:
import requests
from bs4 import BeautifulSoup

In [3]:
import pandas as pd

BASE = "https://www.genie.co.kr/chart/top200"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Referer": "https://www.genie.co.kr/",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

def fetch_page(pg: int):
    res = requests.get(BASE, headers=HEADERS, params={"pg": pg}, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    table = soup.select_one("div.newest-list > div > table")
    if table is None:
        return []
    rows = table.select("tbody > tr")

    items = []
    for idx, tr in enumerate(rows):
        title_el  = tr.select_one("a.title.ellipsis")
        artist_el = tr.select_one("a.artist.ellipsis") or tr.select_one("a.artist")
        if not title_el:
            continue
        rank = (pg - 1) * 50 + (idx + 1)  # 1~100 순위
        title  = title_el.get_text(strip=True)
        artist = artist_el.get_text(strip=True) if artist_el else ""
        items.append({"순위": rank, "곡제목": title, "아티스트": artist})
    return items

# 1~100위 수집
all_items = []
for pg in (1, 2):
    all_items.extend(fetch_page(pg))

# DataFrame으로 변환 (컬럼: 순위/곡제목/아티스트)
df = pd.DataFrame(all_items, columns=["순위", "곡제목", "아티스트"]).sort_values("순위").reset_index(drop=True)

print(df.head(10))   # 확인용

   순위                    곡제목  \
0   1                 Golden   
1   2               Soda Pop   
2   3               뛰어(JUMP)   
3   4                 FAMOUS   
4   5               Drowning   
5   6                 시작의 아이   
6   7              Your Idol   
7   8                너에게 닿기를   
8   9  모르시나요 (Prod. by 로코베리)   
9  10          How It's Done   

                                                아티스트  
0  HUNTR/X & EJAE & Audrey Nuna & REI AMI & KPop ...  
1  Saja Boys & Andrew Choi & Neckwav & Danny Chun...  
2                                          BLACKPINK  
3                                     ALLDAY PROJECT  
4                                              WOODZ  
5                                       마크툽 (Maktub)  
6  Saja Boys & Andrew Choi & Neckwav & Danny Chun...  
7                                               10CM  
8                                                조째즈  
9  HUNTR/X & EJAE & Audrey Nuna & REI AMI & KPop ...  


In [4]:
df

,순위,곡제목,아티스트
0,1,Golden,HUNTR/X & EJAE & Audrey Nuna & REI AMI & KPop ...
1,2,Soda Pop,Saja Boys & Andrew Choi & Neckwav & Danny Chun...
2,3,뛰어(JUMP),BLACKPINK
3,4,FAMOUS,ALLDAY PROJECT
4,5,Drowning,WOODZ
...,...,...,...
95,96,Stay,The Kid LAROI & Justin Bieber
96,97,너의 모든 순간,성시경
97,98,봄여름가을겨울 (Still Life),BIGBANG (빅뱅)
98,99,DRIP,BABYMONSTER
